# Bronze Layer - Confluence Data Extraction

Extracts all spaces, pages, and comments from Confluence Cloud
and writes them as raw Parquet files into the Bronze zone.

In [ ]:
# Parameters (overridden by pipeline at runtime)
confluence_url = ""
confluence_email = ""
confluence_api_token = ""


In [ ]:
# Install Confluence dependencies
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'atlassian-python-api>=3.41.0'])
print('Dependencies installed.')

In [ ]:
import pandas as pd
from datetime import datetime, timezone
from atlassian import Confluence
import os

# ── Connect to Confluence ────────────────────────────────────────
api = Confluence(
    url=confluence_url,
    username=confluence_email,
    password=confluence_api_token,
    cloud=True,
)
print(f'Connected to Confluence: {confluence_url}')

# Lakehouse mount point (auto-mounted by Fabric runtime)
BASE = '/lakehouse/default/Files'

In [ ]:
# ── Extract Spaces ───────────────────────────────────────────────
spaces = []
start, limit = 0, 50
while True:
    batch = api.get_all_spaces(start=start, limit=limit, expand='description.plain')
    results = batch.get('results', [])
    if not results:
        break
    for s in results:
        spaces.append({
            'space_id': s.get('id'),
            'space_key': s.get('key'),
            'space_name': s.get('name'),
            'space_type': s.get('type'),
            'description': s.get('description', {}).get('plain', {}).get('value', ''),
        })
    if batch.get('size', 0) < limit:
        break
    start += limit

spaces_df = pd.DataFrame(spaces)
print(f'Extracted {len(spaces_df)} spaces')

In [ ]:
# ── Extract Pages ────────────────────────────────────────────────
pages = []
for sk in spaces_df['space_key']:
    start, limit = 0, 50
    while True:
        results = api.get_all_pages_from_space(
            sk, start=start, limit=limit,
            expand='body.storage,version,history',
        )
        if not results:
            break
        for page in results:
            body_html = page.get('body', {}).get('storage', {}).get('value', '')
            version = page.get('version', {})
            history = page.get('history', {})
            pages.append({
                'page_id': page.get('id'),
                'space_key': sk,
                'title': page.get('title'),
                'status': page.get('status'),
                'body_html': body_html,
                'version_number': version.get('number'),
                'created_by': history.get('createdBy', {}).get('displayName', ''),
                'created_date': history.get('createdDate', ''),
                'last_updated_by': version.get('by', {}).get('displayName', ''),
                'last_updated_date': version.get('when', ''),
            })
        if len(results) < limit:
            break
        start += limit

pages_df = pd.DataFrame(pages)
print(f'Extracted {len(pages_df)} pages')

In [ ]:
# ── Extract Comments ─────────────────────────────────────────────
all_comments = []
for page_id in pages_df['page_id']:
    start, limit = 0, 50
    while True:
        result = api.get_page_comments(
            str(page_id), start=start, limit=limit,
            expand='body.storage,version',
        )
        items = result.get('results', [])
        if not items:
            break
        for c in items:
            all_comments.append({
                'comment_id': c.get('id'),
                'page_id': str(page_id),
                'body_html': c.get('body', {}).get('storage', {}).get('value', ''),
                'author': c.get('version', {}).get('by', {}).get('displayName', ''),
                'created_date': c.get('version', {}).get('when', ''),
            })
        if len(items) < limit:
            break
        start += limit

comments_df = pd.DataFrame(all_comments) if all_comments else pd.DataFrame()
print(f'Extracted {len(comments_df)} comments')

In [ ]:
# ── Write to Bronze Layer ───────────────────────────────────────
import os
now = datetime.now(timezone.utc).isoformat()

bronze_data = {
    'confluence_spaces': spaces_df,
    'confluence_pages': pages_df,
    'confluence_comments': comments_df,
}

for table_name, df in bronze_data.items():
    if df.empty:
        print(f'  Skipping empty table: {table_name}')
        continue
    df = df.copy()
    df['_ingested_at'] = now
    df['_source_file'] = 'confluence_api'

    out_dir = f'{BASE}/bronze/{table_name}'
    os.makedirs(out_dir, exist_ok=True)
    path = f'{out_dir}/data.parquet'
    df.to_parquet(path, index=False)
    print(f'  Wrote {len(df)} rows to {path}')

print('Bronze layer complete.')